<a href="https://colab.research.google.com/github/rpakdel/bz-basic/blob/main/Run_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from bz_algorithm import Block, BZScheduler
import matplotlib.colors as mcolors

def generate_2d_deposit(width, depth):
    """
    Generates a synthetic 2D block model.
    Value increases with depth (simulating a deep ore body)
    but overburden (waste) is on top.
    """
    blocks = []
    grid = {} # To easily find neighbors

    block_id_counter = 0

    for y in range(depth): # y is depth (0 is surface)
        for x in range(width):
            # Procedural generation of Ore vs Waste
            # Ore is a blob in the middle/bottom

            center_x = width // 2
            center_y = depth // 2 + 2
            dist = np.sqrt((x - center_x)**2 + (y - center_y)**2)

            tonnage = 1000 # fixed tonnages per block

            # Grade distribution
            if dist < width / 4:
                grade = 1.5 + np.random.normal(0, 0.2) # High grade ore
                val_per_ton = (grade * 50) - 20 # Revenue - Processing Cost
                value = val_per_ton * tonnage
                type = "Ore"
            elif dist < width / 2.5:
                grade = 0.6 + np.random.normal(0, 0.1) # Low grade ore
                val_per_ton = (grade * 50) - 20
                value = val_per_ton * tonnage
                type = "LowGrade"
            else:
                grade = 0.0 # Waste
                value = -5 * tonnage # Mining cost only (waste)
                type = "Waste"

            b = Block(block_id_counter, x, y, 0, tonnage, grade, value)
            blocks.append(b)
            grid[(x,y)] = b
            block_id_counter += 1

    # Add Precedence (Slope Constraints)
    # 1-to-3 pattern: To mine (x, y), you need (x-1, y-1), (x, y-1), (x+1, y-1)
    for b in blocks:
        if b.y > 0: # If not on surface
            predecessors_coords = [
                (b.x - 1, b.y - 1),
                (b.x,     b.y - 1),
                (b.x + 1, b.y - 1)
            ]
            for px, py in predecessors_coords:
                if (px, py) in grid:
                    b.add_predecessor(grid[(px, py)])

    return blocks, width, depth

def visualize_results(blocks, schedule, width, depth, history):
    """
    Plots the pit phases and the convergence history.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # 1. Plot the Pit Shells / Phases
    matrix = np.full((depth, width), -1.0) # -1 for unmined

    for b in blocks:
        period = schedule[b.id]
        if period != -1:
            matrix[b.y, b.x] = period # Mine value is the period index

    # Create custom colormap: Grey for unmined, Viridis for periods
    cmap = plt.cm.viridis
    cmap.set_under('lightgrey')

    im = ax1.imshow(matrix, cmap=cmap, vmin=0)
    ax1.set_title("Optimization Result: Extraction Period")
    ax1.set_xlabel("X (Block Coordinates)")
    ax1.set_ylabel("Y (Depth)")
    fig.colorbar(im, ax=ax1, label='Period Mined', ticks=range(10))

    # Annotate blocks with 'O' for ore
    for b in blocks:
        if b.economic_value > 0 and schedule[b.id] != -1:
            ax1.text(b.x, b.y, '.', ha='center', va='center', color='white', fontsize=6)

    # 2. Plot Convergence History
    iterations = [h[0] for h in history]
    profits = [h[1] for h in history]
    violations = [h[2] for h in history]

    ax2.set_title("BZ Convergence History")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("NPV ($)", color='tab:blue')
    ax2.plot(iterations, profits, color='tab:blue', marker='o', label='NPV')
    ax2.tick_params(axis='y', labelcolor='tab:blue')
    ax2.grid(True, linestyle='--', alpha=0.6)

    ax3 = ax2.twinx()
    ax3.set_ylabel("Constraint Violation (Tons)", color='tab:red')
    ax3.plot(iterations, violations, color='tab:red', linestyle='--', label='Violation')
    ax3.tick_params(axis='y', labelcolor='tab:red')

    plt.tight_layout()
    plt.show()

# --- Main Execution ---

if __name__ == "__main__":
    # Parameters
    WIDTH = 25
    DEPTH = 15
    PERIODS = 4
    DISCOUNT_RATE = 0.10

    # Generate Data
    print("Generating Block Model...")
    blocks, w, d = generate_2d_deposit(WIDTH, DEPTH)
    print(f"Model created: {len(blocks)} blocks.")

    # Define Limits (Tuning these makes the problem harder/easier)
    # Estimate total tonnage to set realistic constraints
    total_ore_tonnage = sum(b.tonnage for b in blocks if b.economic_value > 0)

    # Constraint: Force spreading extraction over periods
    # e.g., limit to 30% of total possible ore per period
    MINING_CAP = 30000
    PROC_CAP = total_ore_tonnage / 2.5

    print(f"Constraints :: Mining: {MINING_CAP}t/yr | Processing: {PROC_CAP:.0f}t/yr")

    # Initialize Solver
    scheduler = BZScheduler(
        blocks,
        PERIODS,
        DISCOUNT_RATE,
        MINING_CAP,
        PROC_CAP
    )

    # Run BZ Algorithm
    # Note: Step size factor is sensitive. In real BZ, this is dynamic.
    final_schedule, history = scheduler.solve(max_iterations=15, step_size_factor=0.00008)

    # Visualize
    visualize_results(blocks, final_schedule, w, d, history)